In [1]:
# Install required libraries
#fastapi — the web framework
#uvicorn — the server that runs FastAPI 

import subprocess
subprocess.run(['pip', 'install', 'fastapi', 'uvicorn[standard]'], check=True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 1.8 MB/s  0:00:011.8 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 1.6 MB/s  0:00:02m 1.6 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14/14 [fastapi]━ 13/14 [fastapi]e]antic]tenv]


CompletedProcess(args=['pip', 'install', 'fastapi', 'uvicorn[standard]'], returncode=0)

In [6]:
import os
import numpy as np
import json

# Point to the model folder 
MODEL_DIR = os.path.join(os.getcwd(), '..', 'model')
MODEL_DIR = os.path.abspath(MODEL_DIR)   # converts .. into a clean path
print("Model folder:", MODEL_DIR)
print()

# Check files exist 
required_files = ['model_weights.npz', 'scale_params.json']

for f in required_files:
    full_path = os.path.join(MODEL_DIR, f)
    if os.path.exists(full_path):
        print(f' {f} found')
    else:
        print(f' {f} NOT found at {full_path}')

# Load weights 
print()
weights = np.load(os.path.join(MODEL_DIR, 'model_weights.npz'))
print('Weight shapes:')
for name in weights.files:
    print(f'  {name}: {weights[name].shape}')

# Load scale params 
print()
with open(os.path.join(MODEL_DIR, 'scale_params.json'), 'r') as f:
    scale_params = json.load(f)
print('Features in scale_params.json:')
for feature, bounds in scale_params.items():
    print(f'  {feature}: min={bounds["min"]:.4f}, max={bounds["max"]:.4f}')

Model folder: /home/narjiss/Documents/PFE_Project/model

 model_weights.npz found
 scale_params.json found

Weight shapes:
  W1: (11, 16)
  b1: (1, 16)
  W2: (16, 1)
  b2: (1, 1)

Features in scale_params.json:
  CreditLineUsage: min=0.0000, max=10.0078
  Age: min=21.0000, max=109.0000
  Late30to59Days: min=0.0000, max=4.5951
  DebtRatio: min=0.0000, max=12.6346
  MonthlyIncome: min=0.0000, max=13.6352
  OpenCreditLines: min=0.0000, max=4.0604
  Late90Days: min=0.0000, max=4.5951
  RealEstateLoans: min=0.0000, max=3.4012
  Late60to89Days: min=0.0000, max=4.5951
  Dependents: min=0.0000, max=2.3979


In [7]:
# MinMax scaling function
# the same logic we ipplied during preprocessing in Document3, we gonna re-implement it here
# so the API can scale incoming raw inputs.

def scale_input(raw_input: dict, scale_params: dict) -> np.ndarray:
        # The exact order that the model expects (must match training column order)
        feature_order = [
            'CreditLineUsage', 'Age', 'Late30to59Days', 'DebtRatio',
            'MonthlyIncome', 'OpenCreditLines', 'Late90Days',
            'RealEstateLoans', 'Late60to89Days', 'Dependents',
            'MonthlyIncome_Was_Missing'
        ]

        scaled_values = []
      
        for feature in feature_order:
            raw_value = raw_input[feature]

            # MonthlyIncome_Was_Missing is already 0 or 1 — no scaling needed
            if feature == 'MonthlyIncome_Was_Missing':
                scaled_values.append(float(raw_value))
                continue

            f_min = scale_params[feature]['min']
            f_max = scale_params[feature]['max']

            # Apply MinMax formula
            scaled = (raw_value - f_min) / (f_max - f_min)

            # Clip to [0, 1] to handle values outside the training range
            scaled = float(np.clip(scaled, 0.0, 1.0))
            scaled_values.append(scaled)

        # Return as a 2D array with shape (1, 11)
        # Shape (1, 11) means: 1 sample, 11 features — matches training format
        return np.array(scaled_values).reshape(1, -1)


# ── Quick test ─────────────────────────────────────────────────────────────────
test_raw = {
    'CreditLineUsage': 0.5,
    'Age': 45,
    'Late30to59Days': 0,
    'DebtRatio': 0.3,
    'MonthlyIncome': 8.5,          # already log-transformed as in preprocessing
    'OpenCreditLines': 2.1,
    'Late90Days': 0,
    'RealEstateLoans': 0,
    'Late60to89Days': 0,
    'Dependents': 0,
    'MonthlyIncome_Was_Missing': 0
}

scaled = scale_input(test_raw, scale_params)
print('Scaled input shape:', scaled.shape)   # expect (1, 11)
print('Scaled values:', np.round(scaled, 4))

Scaled input shape: (1, 11)
Scaled values: [[0.05   0.2727 0.     0.0237 0.6234 0.5172 0.     0.     0.     0.
  0.    ]]


In [8]:
# Activation functions 
# Same Forward Propagation we  use on Document2 - same equation, same weights convention


def relu(Z):
    return np.maximum(0, Z)

def sigmoid(Z):
    Z = np.clip(Z, -500, 500)      # prevent overflow with extreme values
    return 1 / (1 + np.exp(-Z))


# Forward propagation (inference) 
def predict(X_scaled, W1, b1, W2, b2, threshold=0.4):

    # Hidden layer — MUST use np.dot(X, W1), not np.dot(W1, X)
    Z1 = np.dot(X_scaled, W1) + b1     # shape: (1, 16)
    A1 = relu(Z1)                       # shape: (1, 16)

    # Output layer
    Z2 = np.dot(A1, W2) + b2           # shape: (1, 1)
    A2 = sigmoid(Z2)                    # shape: (1, 1) — value in (0, 1)

    probability = float(A2[0, 0])
    verdict = 'HIGH RISK' if probability >= threshold else 'LOW RISK'

    return probability, verdict


# Quick test 
W1 = weights['W1']   # shape (11, 16)
b1 = weights['b1']   # shape (1, 16)
W2 = weights['W2']   # shape (16, 1)
b2 = weights['b2']   # shape (1, 1)

prob, verdict = predict(scaled, W1, b1, W2, b2)
print(f'Default probability : {prob:.4f}')
print(f'Risk verdict        : {verdict}')

Default probability : 0.2258
Risk verdict        : LOW RISK
